In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math
import pandas as pd

def h1_star(L1, L2, R1, c1, R2, c2,  mu):
    return (1/L1)*(mu + c2/R2 - c1/R1)

def h2_star(L1, L2, R1, c1, R2, c2, mu):
    return (1/L2)*(-mu*R1*c2/(c1*R2) - c2/R2 + c1/R1)


def intersect(L1, L2, R1, c1, R2, c2,  mu):
    b1 = (R1*mu**2 - L1*c1 - R1*L1*mu)/(L2*c1)
    b2 = (R1*mu**2)/(L2*c1)

    h11 = (R1*mu**2 - L2*c1) / ( L1*c1 + L1*R1*mu)
    h12 = (R1*mu**2) / ( L1*c1 + L1*R1*mu)

    def h2_for_given_h1(h1):
            x_max1 = (c2/(2*R2))*(-1 + math.sqrt(1 + (4*R2*mu/c2)))
            x_max2 = (-(c2 - R2*L2) + math.sqrt( (c2-R2*L2)**2 + (4*R2*c2*(L2+mu)) ) ) / (2*R2)
            h_max = (mu - x_max1)/L1
            h_min = (mu - x_max2)/L1
    
            if h1 >= h_max:
                h2 = 0
            elif h1 <= h_min:
                h2 = 1
            else:
                up = R2*mu**2 -2*R2*mu*L1*h1 + R2*(L1**2)*(h1**2) - L1*c2*h1
                down = L2*c2 + mu*R2*L2 - R2*L1*h1*L2
                h2 = up/down
            return h2

    # case 1 
    if b1 > 1 and b2 > 1:
        sol = (1, h2_for_given_h1(1))

    # case 2 
    elif b1 < 0 and b2 > 1:
        if h2_for_given_h1(h11) == 0:
            sol = (h12, 0)
        elif h2_for_given_h1(h11) == 1:
            sol = (h11, 1)
        else:
            if h2_for_given_h1(h12) == 0:
                sol = (h12, 0)
            else:
                sol = (h1_star(L1, L2, R1, c1, R2, c2, mu), h2_star(L1, L2, R1, c1, R2, c2, mu))

    # case 3
    elif b1 < 0 and 0 <= b2 <= 1:
        if h2_for_given_h1(0) == 1:
            sol = (0,1)
        elif b2 <= h2_for_given_h1(0):
            sol = (0, h2_for_given_h1(0))
        else:
            if h2_for_given_h1(h12) == 0:
                sol = (h12, 0)
            else:
                sol = (h1_star(L1, L2, R1, c1, R2, c2, mu), h2_star(L1, L2, R1, c1, R2, c2, mu))

    # case 4
    elif 0 <= b1 <= 1 and b2 > 1:
        if h2_for_given_h1(h11) == 1:
            sol = (h11,1)
        else:
            if h2_for_given_h1(1) <= b1:
                sol = (1, h2_for_given_h1(1))
            else:
                sol = (h1_star(L1, L2, R1, c1, R2, c2, mu), h2_star(L1, L2, R1, c1, R2, c2, mu))

    # case 5
    elif 0 <= b1 <= 1 and 0 <= b2 <= 1:
        if h2_for_given_h1(0) == 1:
            sol = (0,1)
        elif b2 <= h2_for_given_h1(0):
            sol = (0, h2_for_given_h1(0))
        else:
            if h2_for_given_h1(1) <= b1:
                sol = (1, h2_for_given_h1(1) )
            else:
                sol = (h1_star(L1, L2, R1, c1, R2, c2, mu), h2_star(L1, L2, R1, c1, R2, c2,  mu))

    return sol[0], sol[1]

In [2]:
def W1(L1, L2, R1, c1, R2, c2, mu, h1, h2):
    return (L1*h1 + L2*h2)/(mu*(mu-L1*h1))

In [3]:
def W2(L1, L2, R1, c1, R2, c2, mu, h1, h2):
    return (L1*h1 + L2*h2)/((mu - L1*h1 - L2*h2)*(mu-L1*h1))

In [4]:
def cum_U1(L1, L2, R1, c1, R2, c2, mu, h1, h2):
    return L1*h1*(R1 - c1*W1(L1, L2, R1, c1, R2, c2, mu, h1, h2))

In [5]:
def cum_U2(L1, L2, R1, c1, R2, c2,  mu, h1, h2):
    return L2*h2*(R2 - c2*W2(L1, L2, R1, c1, R2, c2,  mu, h1, h2))

In [6]:
def throughput(L1, L2, R1, c1, R2, c2,  mu, h1, h2):
    return (L1*h1 + L2*h2)/(L1+L2)

In [7]:
def cum_U(L1, L2, R1, c1, R2, c2,  mu, h1, h2):
    return cum_U1(L1, L2, R1, c1, R2, c2,  mu, h1, h2) + cum_U2(L1, L2, R1, c1, R2, c2,  mu, h1, h2)

In [8]:
intersect(1.301, 0.857, 5, 20, 6, 14,  2.770)

(0.8480655905713556, 0.05931544146246591)

In [9]:
def print_result(L1, L2, R1, c1, R2, c2, mu):
    # Equilibrium probabilities
    h1, h2 = intersect(L1, L2, R1, c1, R2, c2,mu)

    # Performance metrics
    W1_val = W1(L1, L2, R1, c1, R2, c2, mu, h1, h2)      # wait time type-1
    W2_val = W2(L1, L2, R1, c1, R2, c2, mu, h1, h2)      # wait time type-2

    cumU1_val = cum_U1(L1, L2, R1, c1, R2, c2, mu, h1, h2)
    cumU2_val = cum_U2(L1, L2, R1, c1, R2, c2, mu, h1, h2)

    thr_val = throughput(L1, L2, R1, c1, R2, c2, mu, h1, h2)
    cumU_val = cum_U(L1, L2, R1, c1, R2, c2,  mu, h1, h2)

    df = pd.DataFrame([{
        "h1*": h1,
        "h2*": h2,
        "W1": W1_val,
        "W2": W2_val,
        "CumU1": cumU1_val,
        "CumU2": cumU2_val,
        "Throughput": thr_val,
        "Total CumU": cumU_val
    }])


    return df


In [10]:
print_result(1.301, 0.857, 5, 20, 6, 14, 2.770)

,h1*,h2*,W1,W2,CumU1,CumU2,Throughput,Total CumU
0,0.848066,0.059315,0.25,0.428571,-9.799569e-16,-9.029814e-17,0.534832,-1.070255e-15
